# FreshCart ELT — Abhinav Harsh, 23BCS006
Fresh real experiments in an isolated database in a provided trial, with the account owner’s permission. New-account signup failed. This shared-account arrangement differs from the assignment’s own-account requirement. Course sample SQL/data and AI assistance are acknowledged. Snowpark PUT uploads each raw file; COPY and all transformation SQL retain the lecture logic.

Evidence preservation: the native Snowflake Workspace download omits stored cell outputs. The stdout below was restored from this notebook’s actual live UI output and its JSON export bundle, captured on 16 November 2026. No numerical results or account identity were altered. Execution counters remain empty as in the native export.

In [ ]:
from pathlib import Path
import os,sys,json
ROOT=Path('/tmp/abhinav-23bcs006')
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)
FILES = {'elt/freshcart.py': '"""FreshCart course baseline, with the required currency and gold FX checks.\nStudent: Abhinav Harsh, 23BCS006. Adapted with AI assistance from the lecture notebook.\nConnection settings from the lecture have deliberately not been copied.\n"""\nimport os\nfrom getpass import getpass\nLANDING_DIR = \'.local/landing\'\nconn = None\n\n# Define deliberately messy sample API data so we can demonstrate cleaning, deduplication, and enrichment.\n# ---- Website orders: flat JSON ----\nMOCK_WEB_ORDERS = [\n    # ORD-1001: email has spaces and capitals, amount has a comma\n    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",\n     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",\n     "updated_at": "2026-09-01 10:02:00"},\n    # ORD-1003: amount is missing (the mobile copy below has it)\n    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "MEERA S",\n     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",\n     "updated_at": "2026-09-02 09:10:00"},\n    # ORD-1002: a dollar order (also sent by the mobile app = duplicate)\n    {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",\n     "order_date": "2026-09-01", "amount": "42.50", "currency": "USD", "pincode": "560034",\n     "updated_at": "2026-09-01 11:15:00"},\n    # ORD-1004: negative amount -> will be rejected\n    {"order_id": "ORD-1004", "cust_email": "rahul.m@gmail.com", "cust_name": "Rahul M",\n     "order_date": "Sep 3, 2026", "amount": "-500", "currency": "INR", "pincode": "110001",\n     "updated_at": "2026-09-03 12:00:00"},\n    # ORD-1005: rupee sign in the amount, lowercase currency, dot after the name\n    {"order_id": "ORD-1005", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R.",\n     "order_date": "2026-09-03", "amount": "₹ 899", "currency": "inr", "pincode": "600041",\n     "updated_at": "2026-09-03 16:45:00"},\n    # ORD-1006: bad email and 5-digit pincode -> will be rejected\n    {"order_id": "ORD-1006", "cust_email": "vikram@@mail", "cust_name": "Vikram",\n     "order_date": "2026-09-04", "amount": 30, "currency": "EUR", "pincode": "99999",\n     "updated_at": "2026-09-04 08:30:00"},\n]\n\n# ---- Mobile app orders: nested JSON, different field names, day-first dates ----\nMOCK_MOBILE_ORDERS = [\n    # ORD-1002 again (same as the web copy)\n    {"orderId": "ORD-1002", "customer": {"email": "arjun.k@yahoo.in", "name": "Arjun K"},\n     "orderDate": "01/09/2026", "total": {"value": "42.50", "currency": "usd"},\n     "deliveryPin": "560034", "updatedAt": "2026-09-01T11:15:00"},\n    # ORD-1003 again, newer and with the amount filled in\n    {"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"},\n     "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"},\n     "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"},\n]\n\n# ---- Customer tier changes (Priya moves from Silver to Gold on 3 Sept) ----\nMOCK_CRM_TIER_EVENTS = [\n    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},\n    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-03"},\n    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},\n    {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},\n]\n\n# ---- Exchange rates: rupees per 1 US dollar, per day ----\nMOCK_FX_RESPONSES = [\n    {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "2026-09-03",\n     "rates": {"2026-09-01": {"INR": 83.20}, "2026-09-02": {"INR": 83.35}, "2026-09-03": {"INR": 83.10}}},\n]\n\n# ---- Pincodes: pincode -> (city, state) ----\nMOCK_PINCODES = {"600041": ("Chennai", "Tamil Nadu"), "560034": ("Bengaluru", "Karnataka"),\n                 "400050": ("Mumbai", "Maharashtra")}\n\n\ndef mock_pincode_response(pin):\n    """Reply in the same shape the India Post API uses."""\n    if pin not in MOCK_PINCODES:\n        return [{"Message": "No records found", "Status": "Error", "PostOffice": None}]\n    district, state = MOCK_PINCODES[pin]\n    return [{"Message": "Number of pincode(s) found:1", "Status": "Success",\n             "PostOffice": [{"Name": district, "District": district, "State": state,\n                             "Country": "India", "Pincode": pin}]}]\n\n# Define raw-file writing so each source is saved with run metadata before any transformations.\nimport gzip, json  # Import tools for gzip compression and JSON conversion.\nfrom datetime import date, datetime, timezone  # Import date/time tools; date is unused here.\nfrom pathlib import Path  # Import Path to work with folders and file paths.\n\n\ndef write_landing(source, records, run_id):  # Define a function to save records from one source.\n    """Save source records in a compressed JSON Lines file and return its path."""\n    folder = Path(LANDING_DIR) / source  # Build the source folder path; LANDING_DIR must already be defined.\n    folder.mkdir(parents=True, exist_ok=True)  # Create the folder and missing parent folders if needed.\n    path = folder / f"{source}__{run_id}.json.gz"  # Create a filename containing the source and run ID.\n    now = datetime.now(timezone.utc).isoformat()  # Capture the current UTC extraction time.\n    with gzip.open(path, "wt", encoding="utf-8") as f:  # Open a compressed file for writing UTF-8 text.\n        for rec in records:  # Process each input record.\n            envelope = {"source": source, "run_id": run_id, "extracted_at": now, "data": rec}  # Attach extraction metadata to the record.\n            f.write(json.dumps(envelope, ensure_ascii=False) + "\\n")  # Write one JSON record per line while preserving Unicode characters.\n    print(f"   {source}: {len(records)} records -> {path}")  # Display the record count and saved file path.\n    return path  # Return the saved file path.\n\n# Define SQL and file-loading helpers so the pipeline can upload raw files and inspect Snowflake tables.\nimport pandas as pd  # Import pandas to display query results as tables.\n\n\ndef run_sql(sql_text):  # Define a helper to execute multiple SQL statements.\n    """Run a SQL script that contains several statements."""\n    conn.execute_string(sql_text, remove_comments=True)  # Execute the SQL script after removing comments.\n\n\ndef query(sql):  # Define a helper to execute SQL and retrieve its results.\n    """Run one query and return its rows."""\n    with conn.cursor() as cur:  # Open a cursor that closes automatically when the block ends.\n        cur.execute(sql)  # Execute the supplied SQL statement.\n        return cur.fetchall()  # Return all result rows as a list of tuples.\n\n\ndef show(sql):  # Define a helper to return SQL results as a pandas table.\n    """Run one query and return the result as a table."""\n    with conn.cursor() as cur:  # Open a cursor to interact with Snowflake.\n        cur.execute(sql)  # Execute the supplied query.\n        return pd.DataFrame(\n            cur.fetchall(),  # Retrieve all result rows.\n            columns=[c[0] for c in cur.description]  # Use the query\'s column names as table headings.\n        )\n\n\ndef load_file(local_path, table, stage_dir):  # Define a helper to upload a file and load its records.\n    """Upload one raw file to Snowflake, then copy its records into a bronze table."""\n    uri = "file://" + Path(local_path).resolve().as_posix()  # Convert the local path into an absolute file URI.\n    query(\n        f"PUT \'{uri}\' @bronze.landing/{stage_dir}/ AUTO_COMPRESS=FALSE OVERWRITE=FALSE"\n    )  # Upload the file to the stage without additional compression or overwriting existing files.\n    rows = query(f"""\n        -- Load records from the specified stage folder into the target table.\n        COPY INTO {table} (source, raw, run_id, extracted_at, src_file, src_row)\n        FROM (\n          -- Extract JSON fields and capture the source filename and row number.\n          SELECT $1:source::STRING, $1:data, $1:run_id::STRING,\n                 $1:extracted_at::TIMESTAMP_TZ, METADATA$FILENAME, METADATA$FILE_ROW_NUMBER\n          -- Read staged files from this folder.\n          FROM @bronze.landing/{stage_dir}/\n        )\n        -- Parse files using the predefined newline-delimited JSON format.\n        FILE_FORMAT = (FORMAT_NAME = \'bronze.ff_ndjson\')\n        -- Abort the COPY statement if a loading error occurs.\n        ON_ERROR = \'ABORT_STATEMENT\'\n    """)  # Execute COPY INTO and collect its loading results.\n    loaded = sum(int(r[3]) for r in rows if len(r) > 3)  # Sum the rows loaded from the fourth column of each COPY result.\n    print(f"   {loaded} rows -> {table}")  # Display the total loaded rows and destination table.\n\n# Store setup SQL for later execution so the warehouse, schemas, stage, and tables are ready before loading.\nSQL_00_SETUP = r"""\n-- =====================================================================\n-- Setup: creates the warehouse, database, schemas and empty tables.\n-- Safe to re-run: everything is IF NOT EXISTS, except the two small seed tables.\n-- =====================================================================\nCREATE WAREHOUSE IF NOT EXISTS ABHINAV_LAB_WH\n  WAREHOUSE_SIZE = \'XSMALL\' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;\nUSE WAREHOUSE ABHINAV_LAB_WH;\n\nCREATE DATABASE IF NOT EXISTS FRESHCART23BCS006;\nUSE DATABASE FRESHCART23BCS006;\n\nCREATE SCHEMA IF NOT EXISTS bronze;      -- raw API responses, unchanged\nCREATE SCHEMA IF NOT EXISTS silver;      -- cleaned, typed, deduplicated\nCREATE SCHEMA IF NOT EXISTS dim;         -- dimensions (customers)\nCREATE SCHEMA IF NOT EXISTS ref;         -- reference data (fx, pincodes, tiers)\nCREATE SCHEMA IF NOT EXISTS gold;        -- business-ready tables\nCREATE SCHEMA IF NOT EXISTS quarantine;  -- rows that failed validation\n\n-- Landing area inside Snowflake for the files Python uploads\nCREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;\nCREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;\n\n-- ---------- BRONZE: one table per source, raw JSON in a VARIANT column ----------\nCREATE TABLE IF NOT EXISTS bronze.orders_raw (\n  ingest_seq   NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER,   -- arrival order\n  source       STRING,          -- \'web\' or \'mobile\'\n  raw          VARIANT,         -- the API record exactly as received\n  run_id       STRING,\n  extracted_at TIMESTAMP_TZ,\n  src_file     STRING,\n  src_row      NUMBER,\n  loaded_at    TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()\n);\nCREATE TABLE IF NOT EXISTS bronze.customers_raw (\n  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,\n  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,\n  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()\n);\nCREATE TABLE IF NOT EXISTS bronze.fx_raw (\n  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,\n  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,\n  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()\n);\nCREATE TABLE IF NOT EXISTS bronze.pincode_raw (\n  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,\n  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,\n  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()\n);\n\n-- ---------- REF: small hand-maintained seed tables ----------\nCREATE OR REPLACE TABLE ref.tier_multiplier AS\nSELECT * FROM VALUES (\'Silver\', 1), (\'Gold\', 2), (\'Platinum\', 3) AS v(tier, multiplier);\n\nCREATE OR REPLACE TABLE ref.state_region AS\nSELECT * FROM VALUES\n  (\'Tamil Nadu\',\'South\'), (\'Karnataka\',\'South\'), (\'Kerala\',\'South\'), (\'Andhra Pradesh\',\'South\'),\n  (\'Telangana\',\'South\'), (\'Puducherry\',\'South\'),\n  (\'Maharashtra\',\'West\'), (\'Gujarat\',\'West\'), (\'Goa\',\'West\'), (\'Rajasthan\',\'West\'),\n  (\'Delhi\',\'North\'), (\'Haryana\',\'North\'), (\'Punjab\',\'North\'), (\'Uttar Pradesh\',\'North\'),\n  (\'Himachal Pradesh\',\'North\'), (\'Jammu and Kashmir\',\'North\'), (\'Uttarakhand\',\'North\'),\n  (\'West Bengal\',\'East\'), (\'Odisha\',\'East\'), (\'Bihar\',\'East\'), (\'Jharkhand\',\'East\'), (\'Assam\',\'East\'),\n  (\'Madhya Pradesh\',\'Central\'), (\'Chhattisgarh\',\'Central\')\n  AS v(state, region);\n\n-- Filled incrementally from the India Post API (see SQL_32_PINCODE_GEO)\nCREATE TABLE IF NOT EXISTS ref.pincode_geo (\n  pincode STRING PRIMARY KEY, city STRING, state STRING, region STRING,\n  updated_at TIMESTAMP_LTZ\n);\n\n-- ---------- QUARANTINE, GOLD ----------\nCREATE TABLE IF NOT EXISTS quarantine.orders (\n  run_id STRING, quarantined_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),\n  ingest_seq NUMBER, source STRING, order_id STRING, reasons ARRAY, row_data VARIANT\n);\n\nCREATE TABLE IF NOT EXISTS gold.fct_orders (\n  order_id STRING PRIMARY KEY, order_date DATE, cust_id STRING, email STRING,\n  tier STRING, multiplier NUMBER, amount NUMBER(12,2), currency STRING,\n  fx_rate NUMBER(12,4), amount_inr NUMBER(14,2), loyalty_points NUMBER,\n  is_high_value BOOLEAN, city STRING, state STRING, region STRING,\n  updated_at TIMESTAMP_NTZ, row_hash NUMBER, published_at TIMESTAMP_LTZ\n);\n"""\n\n# Store cleaning SQL for later execution so web/mobile fields are standardized and invalid orders are quarantined.\nSQL_10_ORDERS_CLEAN = r"""\n-- =====================================================================\n-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.\n-- Input : bronze.orders_raw (raw JSON from \'web\' and \'mobile\')\n-- Output: silver.orders_checked (all rows + reasons), silver.orders_clean (good rows),\n--         quarantine.orders (bad rows, appended once per row)\n-- =====================================================================\nCREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS\nWITH unified AS (\n  -- The two APIs name and nest fields differently; map both to one set of columns.\n  SELECT ingest_seq, source,\n         raw:order_id::STRING   AS order_id,\n         raw:cust_email::STRING AS cust_email,\n         raw:cust_name::STRING  AS cust_name,\n         raw:order_date::STRING AS order_date,\n         raw:amount::STRING     AS amount,\n         raw:currency::STRING   AS currency,\n         raw:pincode::STRING    AS pincode,\n         raw:updated_at::STRING AS updated_at\n  FROM bronze.orders_raw WHERE source = \'web\'\n  UNION ALL\n  SELECT ingest_seq, source,\n         raw:orderId::STRING, raw:customer.email::STRING, raw:customer.name::STRING,\n         raw:orderDate::STRING, raw:total.value::STRING, raw:total.currency::STRING,\n         raw:deliveryPin::STRING, raw:updatedAt::STRING\n  FROM bronze.orders_raw WHERE source = \'mobile\'\n),\ntyped AS (\n  SELECT\n    ingest_seq, source,\n    TRIM(order_id)                                    AS order_id,\n    LOWER(TRIM(cust_email))                           AS email,\n    INITCAP(RTRIM(TRIM(cust_name), \'.\'))              AS name,\n    COALESCE(\n      TRY_TO_DATE(order_date, \'YYYY-MM-DD\'),\n      TRY_TO_DATE(order_date, \'DD/MM/YYYY\'),          -- mobile API sends day first\n      TRY_TO_DATE(order_date, \'MON DD, YYYY\')\n    )                                                 AS order_date,\n    TRY_TO_DECIMAL(REGEXP_REPLACE(amount, \'[^0-9.-]\', \'\'), 12, 2) AS amount,\n    UPPER(TRIM(currency))                             AS currency,\n    TRIM(pincode)                                     AS pincode,\n    TRY_TO_TIMESTAMP_NTZ(updated_at)                  AS updated_at\n  FROM unified\n)\nSELECT *,\n  ARRAY_CONSTRUCT_COMPACT(\n    IFF(order_id IS NULL, \'missing_order_id\', NULL),\n    IFF(email IS NULL OR NOT REGEXP_LIKE(email, \'^[^@\\\\s]+@[^@\\\\s]+\\\\.[a-z]{2,}$\'), \'bad_email\', NULL),\n    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode, \'^[1-9][0-9]{5}$\'), \'bad_pincode\', NULL),\n    IFF(order_date IS NULL, \'unparseable_date\', NULL),\n    IFF(updated_at IS NULL, \'unparseable_updated_at\', NULL),\n    IFF(amount <= 0, \'non_positive_amount\', NULL)          -- NULL amount is allowed (flagged below)\n  ) AS reasons\nFROM typed;\n\n-- Good rows\nCREATE OR REPLACE TABLE silver.orders_clean AS\nSELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing\nFROM silver.orders_checked\nWHERE ARRAY_SIZE(reasons) = 0;\n\n-- Bad rows: kept with their reasons. The NOT EXISTS stops the same bad record being\n-- quarantined again when a later run re-extracts it (same source, order, version).\nINSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, row_data)\nSELECT $run_id, c.ingest_seq, c.source, c.order_id, c.reasons, OBJECT_CONSTRUCT(*)\nFROM silver.orders_checked c\nWHERE ARRAY_SIZE(c.reasons) > 0\n  AND NOT EXISTS (\n        SELECT 1 FROM quarantine.orders q\n        WHERE q.source = c.source\n          AND EQUAL_NULL(q.order_id, c.order_id)\n          AND EQUAL_NULL(q.row_data:UPDATED_AT::TIMESTAMP_NTZ, c.updated_at))   -- key is UPPERCASE: OBJECT_CONSTRUCT(*) names keys after the columns\nQUALIFY ROW_NUMBER() OVER (PARTITION BY c.source, c.order_id, c.updated_at\n                           ORDER BY c.ingest_seq DESC) = 1;\n"""\n\n# Store deduplication SQL for later execution so each order keeps only its newest valid version.\nSQL_20_ORDERS_DEDUP = r"""\n-- =====================================================================\n-- SILVER 2: one row per order_id, newest version wins.\n-- Ties on updated_at are broken by arrival order, so reruns give the same answer.\n-- =====================================================================\nCREATE OR REPLACE TABLE silver.orders_dedup AS\nSELECT *\nFROM silver.orders_clean\nQUALIFY ROW_NUMBER() OVER (\n          PARTITION BY order_id\n          ORDER BY updated_at DESC, ingest_seq DESC\n        ) = 1;\n"""\n\n# Store customer-history SQL for later execution so each order can use the customer tier valid on its date.\nSQL_30_CUSTOMERS_SCD2 = r"""\n-- =====================================================================\n-- DIM: build the SCD Type 2 customer table from the CRM\'s tier-change log.\n-- Each change starts a period; the period ends the day before the next change.\n-- =====================================================================\nCREATE OR REPLACE TABLE dim.customers_scd2 AS\nWITH events AS (\n  SELECT raw:cust_id::STRING                 AS cust_id,\n         LOWER(TRIM(raw:email::STRING))      AS email,\n         INITCAP(TRIM(raw:tier::STRING))     AS tier,\n         TRY_TO_DATE(raw:effective_from::STRING) AS valid_from,\n         loaded_at, ingest_seq\n  FROM bronze.customers_raw\n  -- the same event may be extracted on many runs; keep the latest copy\n  QUALIFY ROW_NUMBER() OVER (PARTITION BY cust_id, valid_from\n                             ORDER BY loaded_at DESC, ingest_seq DESC) = 1\n)\nSELECT cust_id, email, tier, valid_from,\n       COALESCE(DATEADD(day, -1, LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from)),\n                \'9999-12-31\'::DATE)                                       AS valid_to,\n       LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from) IS NULL AS is_current\nFROM events\nWHERE cust_id IS NOT NULL AND valid_from IS NOT NULL;\n"""\n\n# Store daily exchange-rate SQL for later execution so orders can be converted to INR even on missing-rate days.\nSQL_31_FX_DAILY = r"""\n-- =====================================================================\n-- REF: daily FX rates to INR, with no gaps.\n-- 1. Flatten the API\'s {"rates": {"2026-09-01": {"INR": 83.2}, ...}} into rows.\n-- 2. Rates are not published on weekends/holidays: fill each missing day with the\n--    previous published rate, so every order date finds a rate.\n-- 3. Add INR = 1.0 for every day, so rupee orders join like any other.\n-- =====================================================================\nCREATE OR REPLACE TABLE ref.fx_daily AS\nWITH published AS (\n  SELECT r.raw:base::STRING          AS currency,\n         f.key::DATE                 AS rate_date,\n         f.value:INR::NUMBER(12,4)   AS rate_to_inr\n  FROM bronze.fx_raw r, LATERAL FLATTEN(input => r.raw:rates) f\n  QUALIFY ROW_NUMBER() OVER (PARTITION BY currency, rate_date\n                             ORDER BY r.loaded_at DESC, r.ingest_seq DESC) = 1\n),\nbounds AS (\n  SELECT MIN(rate_date) AS d0, GREATEST(MAX(rate_date), CURRENT_DATE()) AS d1 FROM published\n),\nspine AS (                                   -- one row per calendar day d0..d1\n  SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, b.d0) AS d\n  FROM bounds b, TABLE(GENERATOR(ROWCOUNT => 4000))\n  QUALIFY d <= (SELECT d1 FROM bounds)\n),\nfilled AS (\n  SELECT c.currency, s.d AS rate_date,\n         LAST_VALUE(p.rate_to_inr) IGNORE NULLS OVER (\n           PARTITION BY c.currency ORDER BY s.d\n           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rate_to_inr,\n         p.rate_to_inr IS NULL AS is_carried_forward\n  FROM spine s\n  CROSS JOIN (SELECT DISTINCT currency FROM published) c\n  LEFT JOIN published p ON p.currency = c.currency AND p.rate_date = s.d\n)\nSELECT currency, rate_date, rate_to_inr, is_carried_forward FROM filled\nUNION ALL\nSELECT \'INR\', d, 1.0, FALSE FROM spine;\n"""\n\n# Store pincode-lookup SQL for later execution so orders can be enriched with city, state, and sales region.\nSQL_32_PINCODE_GEO = r"""\n-- =====================================================================\n-- REF: add newly looked-up pincodes to ref.pincode_geo.\n-- India Post gives district and state; the company\'s sales region comes from\n-- the ref.state_region seed table.\n-- =====================================================================\nMERGE INTO ref.pincode_geo t\nUSING (\n  SELECT p.pincode, p.city, p.state, sr.region\n  FROM (\n    SELECT raw:pincode::STRING                                  AS pincode,\n           raw:response[0]:PostOffice[0]:District::STRING       AS city,\n           raw:response[0]:PostOffice[0]:State::STRING          AS state,\n           loaded_at, ingest_seq\n    FROM bronze.pincode_raw\n    WHERE raw:response[0]:Status::STRING = \'Success\'\n    QUALIFY ROW_NUMBER() OVER (PARTITION BY pincode ORDER BY loaded_at DESC, ingest_seq DESC) = 1\n  ) p\n  LEFT JOIN ref.state_region sr ON sr.state = p.state\n) s\nON t.pincode = s.pincode\nWHEN MATCHED AND (t.city <> s.city OR t.state <> s.state OR NVL(t.region,\'\') <> NVL(s.region,\'\'))\n  THEN UPDATE SET city = s.city, state = s.state, region = s.region, updated_at = CURRENT_TIMESTAMP()\nWHEN NOT MATCHED\n  THEN INSERT (pincode, city, state, region, updated_at)\n       VALUES (s.pincode, s.city, s.state, s.region, CURRENT_TIMESTAMP());\n"""\n\n# Store enrichment and publishing SQL for later execution so clean orders become business-ready gold records.\nSQL_40_GOLD = r"""\n-- =====================================================================\n-- ENRICH + PUBLISH\n-- 1. silver.orders_enriched: point-in-time joins to customer tier and FX rate,\n--    computed columns, geography. LEFT JOINs so no order is silently lost.\n-- 2. gold.fct_orders: idempotent MERGE. Re-running changes nothing unless a\n--    value actually changed (row_hash).\n-- =====================================================================\nCREATE OR REPLACE TABLE silver.orders_enriched AS\nSELECT o.*,\n       c.cust_id, c.tier,\n       m.multiplier,\n       fx.rate_to_inr                              AS fx_rate,\n       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,\n       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,\n       amount_inr >= 2000                          AS is_high_value,\n       g.city, g.state, g.region\nFROM silver.orders_dedup o\nLEFT JOIN dim.customers_scd2 c\n       ON  c.email = o.email\n       AND o.order_date BETWEEN c.valid_from AND c.valid_to     -- tier ON the order date\nLEFT JOIN ref.tier_multiplier m ON m.tier = c.tier\nLEFT JOIN ref.fx_daily fx\n       ON  fx.currency = o.currency AND fx.rate_date = o.order_date  -- rate OF the order date\nLEFT JOIN ref.pincode_geo g ON g.pincode = o.pincode;\n\nMERGE INTO gold.fct_orders t\nUSING (\n  SELECT order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,\n         amount_inr, loyalty_points, is_high_value, city, state, region, updated_at,\n         HASH(order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,\n              amount_inr, loyalty_points, is_high_value, city, state, region, updated_at) AS row_hash\n  FROM silver.orders_enriched\n) s\nON t.order_id = s.order_id\nWHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET\n  order_date = s.order_date, cust_id = s.cust_id, email = s.email, tier = s.tier,\n  multiplier = s.multiplier, amount = s.amount, currency = s.currency, fx_rate = s.fx_rate,\n  amount_inr = s.amount_inr, loyalty_points = s.loyalty_points, is_high_value = s.is_high_value,\n  city = s.city, state = s.state, region = s.region, updated_at = s.updated_at,\n  row_hash = s.row_hash, published_at = CURRENT_TIMESTAMP()\nWHEN NOT MATCHED THEN INSERT\n  (order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate, amount_inr,\n   loyalty_points, is_high_value, city, state, region, updated_at, row_hash, published_at)\nVALUES\n  (s.order_id, s.order_date, s.cust_id, s.email, s.tier, s.multiplier, s.amount, s.currency,\n   s.fx_rate, s.amount_inr, s.loyalty_points, s.is_high_value, s.city, s.state, s.region,\n   s.updated_at, s.row_hash, CURRENT_TIMESTAMP());\n"""\n\n# Define pipeline orchestration and checks so extraction, loading, transformation, and validation run in order.\n# Check completeness and uniqueness so data problems are visible before using gold results.\ndef quality_checks():\n    """Each of these numbers should be 0."""\n    checks = {\n        "gold amount present but INR amount NULL":\n            "SELECT COUNT(*) FROM gold.fct_orders WHERE amount IS NOT NULL AND amount_inr IS NULL",\n        "rows gained or lost while enriching":\n            "SELECT (SELECT COUNT(*) FROM silver.orders_enriched) - (SELECT COUNT(*) FROM silver.orders_dedup)",\n        "duplicate orders in gold":\n            "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM gold.fct_orders",\n        "orders with no customer match":\n            "SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched",\n        "orders with an amount but no FX rate":\n            "SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched",\n        "orders with no city":\n            "SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched",\n    }\n    for name, sql in checks.items():\n        value = query(sql)[0][0]\n        print(f"   {name:38s} {value}   {\'OK\' if value == 0 else \'PROBLEM\'}")\n    rejected = query("SELECT COUNT_IF(ARRAY_SIZE(reasons) > 0) / NULLIF(COUNT(*), 0) FROM silver.orders_checked")[0][0]\n    print(f"   {\'share of rows rejected\':38s} {float(rejected or 0):.0%}")\n\n\n# Orchestrate the dependent stages so each step receives the output of the preceding step.\ndef run_pipeline():\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")      # a unique name for this run\n    print("run_id:", run_id)\n\n    # EXTRACT: preserve source records in files so ingestion has an auditable raw input.\n    print("1. EXTRACT: sample data -> raw files")\n    web_file    = write_landing("web",    MOCK_WEB_ORDERS,      run_id)\n    mobile_file = write_landing("mobile", MOCK_MOBILE_ORDERS,   run_id)\n    crm_file    = write_landing("crm",    MOCK_CRM_TIER_EVENTS, run_id)\n    fx_file     = write_landing("fx",     MOCK_FX_RESPONSES,    run_id)\n\n    # LOAD: upload files and populate bronze so SQL can transform the raw records.\n    print("2. LOAD: raw files -> bronze tables")\n    load_file(web_file,    "bronze.orders_raw",    f"orders_web/{run_id}")\n    load_file(mobile_file, "bronze.orders_raw",    f"orders_mobile/{run_id}")\n    load_file(crm_file,    "bronze.customers_raw", f"crm/{run_id}")\n    load_file(fx_file,     "bronze.fx_raw",        f"fx/{run_id}")\n\n    # TRANSFORM: clean and deduplicate orders before joining customer and reference data.\n    print("3. TRANSFORM: bronze -> silver -> gold")\n    query(f"SET run_id = \'{run_id}\'")          # SQL_10 stamps rejected rows with $run_id\n    run_sql(SQL_10_ORDERS_CLEAN)\n    run_sql(SQL_20_ORDERS_DEDUP)\n\n    # Pincodes: look up only the ones seen in the orders that we don\'t know yet\n    pins = [row[0] for row in query("""\n        SELECT DISTINCT o.pincode FROM silver.orders_dedup o\n        WHERE NOT EXISTS (SELECT 1 FROM ref.pincode_geo g WHERE g.pincode = o.pincode)""")]\n    print("   new pincodes to look up:", pins)\n    if pins:\n        lookups = [{"pincode": pin, "response": mock_pincode_response(pin)} for pin in pins]\n        pin_file = write_landing("pincode", lookups, run_id)\n        load_file(pin_file, "bronze.pincode_raw", f"pincode/{run_id}")\n\n    # Build customer history and reference lookups before enriching and publishing orders.\n    run_sql(SQL_30_CUSTOMERS_SCD2)\n    run_sql(SQL_31_FX_DAILY)\n    run_sql(SQL_32_PINCODE_GEO)\n    run_sql(SQL_40_GOLD)\n\n    # CHECK: report missing matches and duplicates so the final output can be reviewed.\n    print("4. CHECK")\n    quality_checks()\n    orders, revenue, points = query("SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")[0]\n    print(f"GOLD: {orders} orders, revenue ₹{revenue}, {points} loyalty points")\n\n# Preserve the lecture\'s original rule set for the initial baseline experiment.\nSQL_10_BASELINE = SQL_10_ORDERS_CLEAN\nSQL_10_ORDERS_CLEAN = SQL_10_ORDERS_CLEAN.replace(\n    "IFF(order_id IS NULL, \'missing_order_id\', NULL),",\n    "IFF(currency IS NULL OR currency NOT IN (\'INR\',\'USD\'), \'unsupported_currency\', NULL),\\n        IFF(order_id IS NULL, \'missing_order_id\', NULL),")\n\ndef connect():\n    global conn\n    import snowflake.connector\n    options = dict(account=os.environ[\'SNOWFLAKE_ACCOUNT\'],\n                   user=os.environ[\'SNOWFLAKE_USER\'],role=os.environ.get(\'SNOWFLAKE_ROLE\',\'SYSADMIN\'),\n                   client_store_temporary_credential=False,external_browser_timeout=1800)\n    method = os.environ.get(\'SNOWFLAKE_AUTHENTICATOR\',\'\').lower()\n    if method == \'oauth_authorization_code\':\n        options[\'authenticator\'] = \'oauth_authorization_code\'\n    elif method == \'externalbrowser\':\n        options[\'authenticator\'] = \'externalbrowser\'\n        options[\'disable_console_login\'] = False\n    else:\n        options[\'password\'] = getpass(\'Snowflake password (hidden): \')\n    conn = snowflake.connector.connect(**options)\n    return conn\n', 'elt/run_experiments.py': '"""Run B1–B5 on the student\'s own Snowflake trial; no credentials are saved."""\nimport json,contextlib,io\nfrom pathlib import Path\nfrom elt import freshcart as f\nROOT=Path(__file__).resolve().parents[1]\nE=ROOT/\'evidence\'; E.mkdir(exist_ok=True)\nresults={}\ndef capture(label,fn):\n    out=io.StringIO()\n    with contextlib.redirect_stdout(out): value=fn()\n    log=out.getvalue(); print(log,flush=True)\n    (E/(label+\'.txt\')).write_text(log)\n    results[label]=value\n    (E/\'freshcart-results.json\').write_text(json.dumps(results,indent=2,default=str))\n    return value\n\ndef totals(): return f.query(\'SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders\')[0]\ndef stamp(): return dict(f.query(\'SELECT order_id,published_at FROM gold.fct_orders\'))\ndef baseline():\n    original=f.SQL_10_ORDERS_CLEAN; f.SQL_10_ORDERS_CLEAN=f.SQL_10_BASELINE\n    f.run_sql(f.SQL_00_SETUP); f.run_pipeline()\n    counts={name:f.query(\'SELECT COUNT(*) FROM \'+name)[0][0] for name in [\'bronze.orders_raw\',\'silver.orders_clean\',\'quarantine.orders\',\'silver.orders_dedup\']}\n    print(\'B1 counts\',counts);print(\'Gold\',totals())\n    assert list(counts.values())==[8,6,2,4] and tuple(map(float,totals()))==(4,7785,126)\n    f.SQL_10_ORDERS_CLEAN=original\n    return {\'counts\':counts,\'gold\':totals()}\ndef trace():\n    raw=f.query("SELECT source,raw FROM bronze.orders_raw WHERE COALESCE(raw:order_id::STRING,raw:orderId::STRING)=\'ORD-1003\' ORDER BY source")\n    gold=f.query("SELECT * FROM gold.fct_orders WHERE order_id IN (\'ORD-1001\',\'ORD-1003\',\'ORD-1005\') ORDER BY order_id")\n    print(\'B2 raw\',raw);print(\'B2 gold\',gold)\n    return {\'raw\':raw,\'gold\':gold}\ndef validation():\n    before=f.query("SELECT source,reasons FROM silver.orders_checked WHERE order_id=\'ORD-1006\'")\n    f.run_pipeline()\n    after=f.query("SELECT DISTINCT source,reasons FROM silver.orders_checked WHERE order_id=\'ORD-1006\'")\n    q=f.query("SELECT COUNT(*) FROM quarantine.orders WHERE order_id=\'ORD-1006\'")[0][0]\n    print(\'BEFORE\',before);print(\'AFTER\',after);print(\'Quarantine rows\',q);print(\'Gold\',totals())\n    assert q==1 and tuple(map(float,totals()))==(4,7785,126)\n    return {\'before\':before,\'after\':after,\'quarantine_rows\':q,\'gold\':totals()}\ndef history():\n    print(\'PRE-RUN EXPECTATIONS (committed before experiments):\', (ROOT/\'predictions.json\').read_text())\n    previous=stamp()\n    f.MOCK_CRM_TIER_EVENTS.append({\'cust_id\':\'CUST-02\',\'email\':\'arjun.k@yahoo.in\',\'tier\':\'Gold\',\'effective_from\':\'2026-08-15\'})\n    f.run_pipeline()\n    rows=f.query("SELECT * FROM dim.customers_scd2 WHERE cust_id=\'CUST-02\' ORDER BY valid_from")\n    order=f.query("SELECT tier,loyalty_points FROM gold.fct_orders WHERE order_id=\'ORD-1002\'")[0]\n    changed=sum(previous[k]!=v for k,v in stamp().items())\n    print(\'SCD2\',rows);print(\'ORD-1002\',order);print(\'Gold\',totals());print(\'Changed published_at\',changed)\n    assert len(rows)==2 and order==(\'Gold\',70) and float(totals()[2])==161 and changed==1\n    prior=stamp(); bronze=f.query(\'SELECT COUNT(*) FROM bronze.orders_raw\')[0][0]\n    f.run_pipeline(); growth=f.query(\'SELECT COUNT(*) FROM bronze.orders_raw\')[0][0]-bronze\n    changed_again=sum(prior[k]!=v for k,v in stamp().items())\n    print(\'IDENTICAL RE-RUN bronze growth\',growth,\'published_at changed\',changed_again)\n    assert growth==8 and changed_again==0\n    return {\'history\':rows,\'order\':order,\'gold\':totals(),\'changed\':changed,\'rerun_bronze_growth\':growth,\'rerun_published_changed\':changed_again}\ndef quality():\n    injected={\'order_id\':\'ORD-FX-TEST\',\'cust_email\':\'arjun.k@yahoo.in\',\'cust_name\':\'Arjun K\',\'order_date\':\'2026-12-31\',\'amount\':\'100\',\'currency\':\'USD\',\'pincode\':\'560034\',\'updated_at\':\'2026-12-31 10:00:00\'}\n    f.MOCK_WEB_ORDERS.append(injected);f.run_pipeline()\n    fail=f.query(\'SELECT COUNT(*) FROM gold.fct_orders WHERE amount IS NOT NULL AND amount_inr IS NULL\')[0][0]\n    print(\'B5 deliberate failing check:\',fail,\'PROBLEM\');assert fail==1\n    f.MOCK_WEB_ORDERS.remove(injected)\n    # Only remove this named disposable mock record, including append-only input history.\n    f.query("DELETE FROM bronze.orders_raw WHERE raw:order_id::STRING=\'ORD-FX-TEST\'")\n    f.query("DELETE FROM gold.fct_orders WHERE order_id=\'ORD-FX-TEST\'")\n    f.run_pipeline(); final=f.query(\'SELECT COUNT(*) FROM gold.fct_orders WHERE amount IS NOT NULL AND amount_inr IS NULL\')[0][0]\n    print(\'B5 restored:\',final,\'OK\');assert final==0\n    return {\'failing_value\':fail,\'restored_value\':final,\'final_gold\':totals()}\nif __name__==\'__main__\':\n    f.connect()\n    try:\n        for label,fn in [(\'B1-baseline\',baseline),(\'B2-trace\',trace),(\'B3-currency\',validation),(\'B4-history\',history),(\'B5-quality\',quality)]:capture(label,fn)\n    finally:\n        try: f.query(\'ALTER WAREHOUSE ABHINAV_LAB_WH SUSPEND\')\n        except Exception: pass\n        f.conn.close()\n', 'elt/snowpark_adapter.py': '"""Run the same sample ELT through a signed-in Snowflake Notebook session.\nPUT uses Snowpark\'s official file API; COPY and transformation SQL are unchanged.\nNo passwords, tokens or owner identity are embedded in this adapter.\n"""\nfrom pathlib import Path\nfrom io import StringIO\nfrom snowflake.connector.util_text import split_statements\n\ndef attach(module,session):\n    def query(sql):\n        return [tuple(row) for row in session.sql(sql).collect()]\n    def run_sql(text):\n        for statement,_ in split_statements(StringIO(text),remove_comments=True):\n            query(statement)\n    def load_file(local_path,table,stage_dir):\n        result=session.file.put(str(Path(local_path).resolve()),f\'@bronze.landing/{stage_dir}/\',auto_compress=False,overwrite=False)\n        print(\'   PUT\',Path(local_path).name,result[0].status)\n        rows=query(f"""COPY INTO {table} (source,raw,run_id,extracted_at,src_file,src_row)\nFROM (SELECT $1:source::STRING,$1:data,$1:run_id::STRING,\n$1:extracted_at::TIMESTAMP_TZ,METADATA$FILENAME,METADATA$FILE_ROW_NUMBER\nFROM @bronze.landing/{stage_dir}/)\nFILE_FORMAT=(FORMAT_NAME=\'bronze.ff_ndjson\') ON_ERROR=\'ABORT_STATEMENT\'""")\n        loaded=sum(int(row[3]) for row in rows if len(row)>3)\n        print(f\'   {loaded} rows -> {table}\')\n    module.query=query;module.run_sql=run_sql;module.load_file=load_file\n', 'predictions.json': '{\n  "recorded_before_experiments_utc": "2026-11-16T14:29:54.143392+00:00",\n  "preparation": "AI-assisted pre-run expectations for student review",\n  "B4": {\n    "history_rows_CUST02": 2,\n    "ORD1002_tier": "Gold",\n    "ORD1002_points": 70,\n    "gold_total_points": 161,\n    "changed_published_at_rows": 1\n  },\n  "D1": {\n    "bangalore.koramangala.food": [\n      "Ravi",\n      "Meena",\n      "Control"\n    ],\n    "bangalore.indiranagar.ride": [\n      "Priya",\n      "Arjun",\n      "Control"\n    ],\n    "bangalore.whitefield.ride": [\n      "Arjun",\n      "Control"\n    ],\n    "chennai.tnagar.food": [\n      "Karthik"\n    ],\n    "chennai.whitefield.ride": [],\n    "bangalore.koramangala": [\n      "Control"\n    ]\n  }\n}\n'}
for name,text in FILES.items():
    path=ROOT/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(text)
sys.path.insert(0,str(ROOT))
from snowflake.snowpark.context import get_active_session
session=get_active_session()
from elt import freshcart as f
from elt.snowpark_adapter import attach
attach(f,session)
from elt.run_experiments import baseline,trace,validation,history,quality,capture,results
print('Execution: signed-in Snowflake Notebook; separate FRESHCART23BCS006 / ABHINAV_LAB_WH')
print('Shared trial used with owner permission after new-account signup failed; own-account requirement exception disclosed.')

Execution: signed-in Snowflake Notebook; separate FRESHCART23BCS006 / ABHINAV_LAB_WH
Shared trial used with owner permission after new-account signup failed; own-account requirement exception disclosed.


## B1-baseline

In [ ]:
capture('B1-baseline', baseline)

run_id: 20261116_160039_165052
1. EXTRACT: sample data -> raw files
   web: 6 records -> .local/landing/web/web__20261116_160039_165052.json.gz
   mobile: 2 records -> .local/landing/mobile/mobile__20261116_160039_165052.json.gz
   crm: 4 records -> .local/landing/crm/crm__20261116_160039_165052.json.gz
   fx: 1 records -> .local/landing/fx/fx__20261116_160039_165052.json.gz
2. LOAD: raw files -> bronze tables
   PUT web__20261116_160039_165052.json.gz UPLOADED
   6 rows -> bronze.orders_raw
   PUT mobile__20261116_160039_165052.json.gz UPLOADED
   2 rows -> bronze.orders_raw
   PUT crm__20261116_160039_165052.json.gz UPLOADED
   4 rows -> bronze.customers_raw
   PUT fx__20261116_160039_165052.json.gz UPLOADED
   1 rows -> bronze.fx_raw
3. TRANSFORM: bronze -> silver -> gold
   new pincodes to look up: ['600041', '560034', '400050']
   pincode: 3 records -> .local/landing/pincode/pincode__20261116_160039_165052.json.gz
   PUT pincode__20261116_160039_165052.json.gz UPLOADED
   3 rows -

## B2-trace

In [ ]:
capture('B2-trace', trace)

B2 raw [('mobile', '{\n  "customer": {\n    "email": "Meera@Outlook.com",\n    "name": "Meera S"\n  },\n  "deliveryPin": "400050",\n  "orderDate": "02/09/2026",\n  "orderId": "ORD-1003",\n  "total": {\n    "currency": "INR",\n    "value": "2,100.00"\n  },\n  "updatedAt": "2026-09-02T18:40:00"\n}'), ('web', '{\n  "amount": null,\n  "currency": "INR",\n  "cust_email": "meera@outlook.com",\n  "cust_name": "MEERA S",\n  "order_date": "2026-09-02",\n  "order_id": "ORD-1003",\n  "pincode": "400050",\n  "updated_at": "2026-09-02 09:10:00"\n}')]
B2 gold [('ORD-1001', datetime.date(2026, 9, 1), 'CUST-01', 'priya.r@gmail.com', 'Silver', 1, Decimal('1250.00'), 'INR', Decimal('1.0000'), Decimal('1250.00'), 12, False, 'Chennai', 'Tamil Nadu', 'South', datetime.datetime(2026, 9, 1, 10, 2), 4284627330879666653, datetime.datetime(2026, 11, 16, 8, 0, 47, 773000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>)), ('ORD-1003', datetime.date(2026, 9, 2), 'CUST-03', 'meera@outlook.com', 'P

## B3-currency

In [ ]:
capture('B3-currency', validation)

run_id: 20261116_160050_156905
1. EXTRACT: sample data -> raw files
   web: 6 records -> .local/landing/web/web__20261116_160050_156905.json.gz
   mobile: 2 records -> .local/landing/mobile/mobile__20261116_160050_156905.json.gz
   crm: 4 records -> .local/landing/crm/crm__20261116_160050_156905.json.gz
   fx: 1 records -> .local/landing/fx/fx__20261116_160050_156905.json.gz
2. LOAD: raw files -> bronze tables
   PUT web__20261116_160050_156905.json.gz UPLOADED
   6 rows -> bronze.orders_raw
   PUT mobile__20261116_160050_156905.json.gz UPLOADED
   2 rows -> bronze.orders_raw
   PUT crm__20261116_160050_156905.json.gz UPLOADED
   4 rows -> bronze.customers_raw
   PUT fx__20261116_160050_156905.json.gz UPLOADED
   1 rows -> bronze.fx_raw
3. TRANSFORM: bronze -> silver -> gold
   new pincodes to look up: []
4. CHECK
   gold amount present but INR amount NULL 0   OK
   rows gained or lost while enriching    0   OK
   duplicate orders in gold               0   OK
   orders with no customer

## B4-history

In [ ]:
capture('B4-history', history)

PRE-RUN EXPECTATIONS (committed before experiments): {
  "recorded_before_experiments_utc": "2026-11-16T14:29:54.143392+00:00",
  "preparation": "AI-assisted pre-run expectations for student review",
  "B4": {
    "history_rows_CUST02": 2,
    "ORD1002_tier": "Gold",
    "ORD1002_points": 70,
    "gold_total_points": 161,
    "changed_published_at_rows": 1
  },
  "D1": {
    "bangalore.koramangala.food": [
      "Ravi",
      "Meena",
      "Control"
    ],
    "bangalore.indiranagar.ride": [
      "Priya",
      "Arjun",
      "Control"
    ],
    "bangalore.whitefield.ride": [
      "Arjun",
      "Control"
    ],
    "chennai.tnagar.food": [
      "Karthik"
    ],
    "chennai.whitefield.ride": [],
    "bangalore.koramangala": [
      "Control"
    ]
  }
}

run_id: 20261116_160058_933524
1. EXTRACT: sample data -> raw files
   web: 6 records -> .local/landing/web/web__20261116_160058_933524.json.gz
   mobile: 2 records -> .local/landing/mobile/mobile__20261116_160058_933524.json.gz


## B5-quality

In [ ]:
capture('B5-quality', quality)

run_id: 20261116_160116_862837
1. EXTRACT: sample data -> raw files
   web: 7 records -> .local/landing/web/web__20261116_160116_862837.json.gz
   mobile: 2 records -> .local/landing/mobile/mobile__20261116_160116_862837.json.gz
   crm: 5 records -> .local/landing/crm/crm__20261116_160116_862837.json.gz
   fx: 1 records -> .local/landing/fx/fx__20261116_160116_862837.json.gz
2. LOAD: raw files -> bronze tables
   PUT web__20261116_160116_862837.json.gz UPLOADED
   7 rows -> bronze.orders_raw
   PUT mobile__20261116_160116_862837.json.gz UPLOADED
   2 rows -> bronze.orders_raw
   PUT crm__20261116_160116_862837.json.gz UPLOADED
   5 rows -> bronze.customers_raw
   PUT fx__20261116_160116_862837.json.gz UPLOADED
   1 rows -> bronze.fx_raw
3. TRANSFORM: bronze -> silver -> gold
   new pincodes to look up: []
4. CHECK
   gold amount present but INR amount NULL 1   PROBLEM
   rows gained or lost while enriching    0   OK
   duplicate orders in gold               0   OK
   orders with no cus

## Preserve execution output and stop the query warehouse

In [ ]:
bundle={'results':results,'logs':{p.stem:p.read_text() for p in (ROOT/'evidence').glob('B*.txt')}}
print('EXPORT_BUNDLE_START\n'+json.dumps(bundle,default=str)+'\nEXPORT_BUNDLE_END')
f.query('ALTER WAREHOUSE ABHINAV_LAB_WH SUSPEND')
print('Assignment query warehouse suspended. Stop the notebook service after exporting.')

EXPORT_BUNDLE_START
{"results": {"B1-baseline": {"counts": {"bronze.orders_raw": 8, "silver.orders_clean": 6, "quarantine.orders": 2, "silver.orders_dedup": 4}, "gold": [4, "7785.00", 126]}, "B2-trace": {"raw": [["mobile", "{\n  \"customer\": {\n    \"email\": \"Meera@Outlook.com\",\n    \"name\": \"Meera S\"\n  },\n  \"deliveryPin\": \"400050\",\n  \"orderDate\": \"02/09/2026\",\n  \"orderId\": \"ORD-1003\",\n  \"total\": {\n    \"currency\": \"INR\",\n    \"value\": \"2,100.00\"\n  },\n  \"updatedAt\": \"2026-09-02T18:40:00\"\n}"], ["web", "{\n  \"amount\": null,\n  \"currency\": \"INR\",\n  \"cust_email\": \"meera@outlook.com\",\n  \"cust_name\": \"MEERA S\",\n  \"order_date\": \"2026-09-02\",\n  \"order_id\": \"ORD-1003\",\n  \"pincode\": \"400050\",\n  \"updated_at\": \"2026-09-02 09:10:00\"\n}"]], "gold": [["ORD-1001", "2026-09-01", "CUST-01", "priya.r@gmail.com", "Silver", 1, "1250.00", "INR", "1.0000", "1250.00", 12, false, "Chennai", "Tamil Nadu", "South", "2026-09-01 10:02:00